# eRetail AI Intelligence Platform
## Notebook 05: Forecasting Engine Benchmarking, Rolling-Origin Backtesting & Model Selection

**Objective**: Execute multi-model rolling-origin backtesting without temporal leakage, benchmark Machine Learning (LightGBM) against heuristic and statistical baselines, evaluate TimesFM foundation model platform availability, and implement segment-level champion routing.

### Core Principles:
- **No Free Lunch**: TimesFM-3 and LightGBM must compete on equal footing with classical baselines.
- **Strict Leakage Protection**: Backtest folds strictly enforce historical cutoff dates.
- **Multi-Level Evaluation**: WAPE, MAE, RMSE, and Bias evaluated globally, by SKU, and by intermittency segment.


In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd

workspace_dir = Path.cwd().resolve()
if workspace_dir.name == "notebooks":
    workspace_dir = workspace_dir.parent
sys.path.insert(0, str(workspace_dir / "src"))

mpl_cache = workspace_dir / ".gemini" / "scratch" / "mpl"
mpl_cache.mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(mpl_cache)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="tab10")
print("Benchmarking environment ready.")


Benchmarking environment ready.


In [2]:
from commerce_ai.data.loaders import CommerceDataLoader
from commerce_ai.analytics.demand import build_daily_demand
from commerce_ai.analytics.features import build_demand_features

data_dir = workspace_dir / "data" / "sample"
loader = CommerceDataLoader()
sales_df = loader.load_csv(data_dir / "sales.csv", validate=False)
inventory_df = loader.load_csv(data_dir / "inventory.csv", validate=False)

# Select 10 diverse SKUs for backtesting
sample_skus = sales_df["sku_id"].value_counts().head(10).index.tolist()
daily_demand = build_daily_demand(
    sales=sales_df,
    inventory=inventory_df,
    skus=sample_skus,
    warehouse_ids=["WH_NORTH_01"],
)

# Build feature dataset with calendar, lags, rolling stats, and stockout masking
features_df = build_demand_features(
    daily_demand,
    include_lags=True,
    include_rolling=True,
    include_calendar=True,
    include_intermittency=True,
)
print(f"Engineered feature dataset: {len(features_df):,} rows across {len(sample_skus)} SKUs.")


Engineered feature dataset: 7,300 rows across 10 SKUs.


In [3]:
from commerce_ai.forecasting.timesfm import TimesFMForecastModel

timesfm_probe = TimesFMForecastModel()
print("TimesFM Foundation Model Status Check:")
print(f" - Is Available in Current Environment: {timesfm_probe.is_available}")
if not timesfm_probe.is_available:
    print(f" - Diagnostic Status: TIMESFM_UNAVAILABLE")
    print(f" - Root Cause: {timesfm_probe._init_error}")
    print(" - Architectural Note: Per Section 18 Master Prompt, TimesFM is treated with zero fabrication.")


TimesFM Foundation Model Status Check:
 - Is Available in Current Environment: False
 - Diagnostic Status: TIMESFM_UNAVAILABLE
 - Root Cause: Google TimesFM is unavailable in this environment: No module named 'timesfm'. TimesFM requires Linux x86_64 TPU/GPU with paxml.
 - Architectural Note: Per Section 18 Master Prompt, TimesFM is treated with zero fabrication.


In [4]:
from commerce_ai.forecasting.config import BacktestConfig
from commerce_ai.forecasting.backtesting import RollingOriginBacktester
from commerce_ai.forecasting.baselines import NaiveModel, SeasonalNaiveModel, MovingAverageModel
from commerce_ai.forecasting.croston import CrostonModel
from commerce_ai.forecasting.exponential_smoothing import ExponentialSmoothingModel
from commerce_ai.forecasting.lightgbm_model import LightGBMForecastModel

# Select 4 representative SKUs for multi-model cross validation
backtest_skus = sample_skus[:4]
target_wh = "WH_NORTH_01"
series_subset = [(s, target_wh) for s in backtest_skus]

# Configure 3 rolling-origin folds with 14-day forecast horizon
backtest_cfg = BacktestConfig(
    horizon=14,
    folds=3,
    step_size=14,
    min_history=90,
)

backtester = RollingOriginBacktester(config=backtest_cfg)
backtester.add_model("Naive", lambda: NaiveModel())
backtester.add_model("Seasonal Naive", lambda: SeasonalNaiveModel(seasonal_period=7))
backtester.add_model("Moving Average (7d)", lambda: MovingAverageModel(window=7))
backtester.add_model("Moving Average (14d)", lambda: MovingAverageModel(window=14))
backtester.add_model("Exponential Smoothing", lambda: ExponentialSmoothingModel(seasonal_periods=7))
backtester.add_model("Croston (SBA)", lambda: CrostonModel(variant="sba"))
backtester.add_model("LightGBM", lambda: LightGBMForecastModel())
backtester.add_model("TimesFM", lambda: TimesFMForecastModel())

print(f"Running rolling-origin backtest on {len(series_subset)} series across {backtest_cfg.folds} folds (14-day horizon)...")
result = backtester.run(
    features_df,
    series_keys=series_subset,
    target_col="units_sold",
    date_col="date",
)

print(f"Backtest completed: {result.total_evaluations} forecast records evaluated.")
print(f"Total failures encountered: {result.total_failures} (Failure Rate: {result.failure_rate:.1%})")


Running rolling-origin backtest on 4 series across 3 folds (14-day horizon)...
Backtest completed: 1344 forecast records evaluated.
Total failures encountered: 12 (Failure Rate: 12.5%)


In [5]:
from commerce_ai.forecasting.evaluation import ForecastEvaluator

evaluator = ForecastEvaluator()
benchmark_table = evaluator.generate_benchmark_table(
    result.predictions_df,
    result.diagnostics_df,
)

print("=" * 85)
print("MASTER MODEL BENCHMARK COMPARISON TABLE (SECTION 25)")
print("=" * 85)
print(benchmark_table.to_string(index=False))
print("=" * 85)


MASTER MODEL BENCHMARK COMPARISON TABLE (SECTION 25)
                Model    MAE    RMSE  WAPE (%)  Bias (%)  Runtime (s)  Failures      Status
             LightGBM 4.1474  6.7143     47.50      4.15       0.4778         0     SUCCESS
Exponential Smoothing 4.3535  7.0508     49.86    -10.42       0.0228         0     SUCCESS
        Croston (SBA) 4.3898  7.1494     50.27    -12.05       0.0062         0     SUCCESS
  Moving Average (7d) 4.5791  7.3323     52.44    -12.47       0.0046         0     SUCCESS
       Seasonal Naive 4.6131  7.3017     52.83    -19.29       0.0032         0     SUCCESS
                Naive 4.7798  7.3302     54.74    -38.92       0.0047         0     SUCCESS
 Moving Average (14d) 4.9158  7.4115     56.30     -1.77       0.0045         0     SUCCESS
              TimesFM 8.7321 10.9547    100.00   -100.00       0.0074        12 UNAVAILABLE


In [6]:
sku_metadata = features_df[["sku_id", "warehouse_id", "intermittency_class"]].drop_duplicates()

hierarchical_eval = evaluator.evaluate_hierarchical(
    result.predictions_df,
    entity_metadata=sku_metadata,
)

if "by_intermittency" in hierarchical_eval:
    print("Performance by Demand Intermittency Segment:")
    int_eval = hierarchical_eval["by_intermittency"]
    cols_to_show = ["model_name", "intermittency_class", "mae", "wape", "bias", "count"]
    print(int_eval[cols_to_show].sort_values(["intermittency_class", "wape"]).to_string(index=False))


Performance by Demand Intermittency Segment:
           model_name intermittency_class    mae   wape    bias  count
             LightGBM             Regular 4.1474  47.50    4.15    168
Exponential Smoothing             Regular 4.3535  49.86  -10.42    168
        Croston (SBA)             Regular 4.3898  50.27  -12.05    168
  Moving Average (7d)             Regular 4.5791  52.44  -12.47    168
       Seasonal Naive             Regular 4.6131  52.83  -19.29    168
                Naive             Regular 4.7798  54.74  -38.92    168
 Moving Average (14d)             Regular 4.9158  56.30   -1.77    168
              TimesFM             Regular 8.7321 100.00 -100.00    168


In [7]:
from commerce_ai.forecasting.selection import ModelSelector, SelectionCriteria

# Select champion with 2.0% parsimony margin towards simpler baselines
selector = ModelSelector(SelectionCriteria(prefer_simpler_baseline_if_within_pct=2.0))
global_winner = selector.select_best_model(benchmark_table)
print(f"Global Champion Model Selected: {global_winner}")

policy = selector.build_segment_policy(
    global_benchmark=benchmark_table,
    segment_eval_df=hierarchical_eval.get("by_intermittency"),
    segment_col="intermittency_class",
)

print("\nChampion Routing Policy by Segment:")
for seg, m in policy.segment_champions.items():
    print(f" - Segment '{seg}': routed to '{m}'")


Global Champion Model Selected: LightGBM

Champion Routing Policy by Segment:
 - Segment 'Regular': routed to 'LightGBM'


In [8]:
from commerce_ai.forecasting.service import ForecastService

service = ForecastService()
batch_sample = features_df[
    (features_df["sku_id"].isin(backtest_skus[:2])) & 
    (features_df["warehouse_id"] == target_wh)
].copy()

batch_forecasts = service.forecast_batch(
    dataset=batch_sample,
    horizon=14,
    policy=policy,
)

print(f"Generated {len(batch_forecasts)} production forecast records via ForecastService:")
summary_cols = ["forecast_date", "sku_id", "warehouse_id", "forecast_units", "model_name", "lower_bound", "upper_bound"]
print(batch_forecasts[summary_cols].head(10).to_string(index=False))


Generated 28 production forecast records via ForecastService:
forecast_date    sku_id warehouse_id  forecast_units model_name  lower_bound  upper_bound
   2025-12-27 SKU_00096  WH_NORTH_01        6.609408   LightGBM     4.695727     8.523089
   2025-12-28 SKU_00096  WH_NORTH_01        5.843654   LightGBM     3.929973     7.757335
   2025-12-29 SKU_00096  WH_NORTH_01        5.385116   LightGBM     3.471435     7.298798
   2025-12-30 SKU_00096  WH_NORTH_01        6.215696   LightGBM     4.302015     8.129377
   2025-12-31 SKU_00096  WH_NORTH_01        6.404783   LightGBM     4.491102     8.318464
   2026-01-01 SKU_00096  WH_NORTH_01        3.111752   LightGBM     1.198071     5.025434
   2026-01-02 SKU_00096  WH_NORTH_01        5.685549   LightGBM     3.771868     7.599230
   2026-01-03 SKU_00096  WH_NORTH_01        4.843026   LightGBM     2.929345     6.756708
   2026-01-04 SKU_00096  WH_NORTH_01        5.042578   LightGBM     3.128897     6.956259
   2026-01-05 SKU_00096  WH_NORTH_01  